In [ ]:
import os
import sys
sys.path.append('..')

import json
import torch
from src.training.constants import *
import transformers

model_path = 'Qwen/Qwen2-VL-7B-Instruct'
device = "cuda"
ori_model = transformers.Qwen2VLForConditionalGeneration.from_pretrained(model_path, device_map=None)

In [ ]:
from src.training.model_file.modeling_pre import *

k = 12
replace_qwen_training_modality_adaptive()
cfg = Qwen2VLConfig(**json.load(open('Qwen/Qwen2-VL-7B-Instruct/config.json')))
new_model = Qwen2VLForConditionalGeneration(cfg, layer=k)

In [ ]:
for name, module in ori_model.named_modules():
    print(name)

In [ ]:
ori_dict = ori_model.state_dict()
new_dict = new_model.state_dict()

for k in ori_dict.keys():
    if k in new_dict:
        new_dict[k] = ori_dict[k]
new_model.load_state_dict(new_dict)

for k in ori_dict.keys():
    # print(k)
    fusion_k = 'fusion.' + '.'.join(k.split('.')[1:])
    # print(fusion_k)
    if fusion_k in new_dict:
        print(fusion_k)
        new_dict[fusion_k] = ori_dict[k]
new_model.load_state_dict(new_dict)

In [ ]:
# keep last layer, we found that when k is small keep last layer will get better performance

# ori_dict = ori_model.state_dict()
# new_dict = new_model.state_dict()

# for k in ori_dict.keys():
#     if k in new_dict:
#         new_dict[k] = ori_dict[k]
# new_model.load_state_dict(new_dict)

# for k in ori_dict.keys():
#     fusion_k = 'fusion.' + '.'.join(k.split('.')[1:])
#     # fusion_k = fusion_k.split('.')[2]
#     parts = fusion_k.split('.')
#     if len(parts) > 2:
#         third_part = parts[2]
#         if third_part.isdigit() and 'block' not in fusion_k:
#             num = int(third_part)
#             if 0 <= num < 11:
#                 if fusion_k in new_dict:
#                     # print(fusion_k)
#                     new_dict[fusion_k] = ori_dict[k]
#             if num == 27:
#                 num = 11
#                 parts[2] = str(num)
#                 fusion_k_ = ".".join(parts)
#                 if fusion_k_ in new_dict:
#                     new_dict[fusion_k_] = ori_dict[k]
#         elif fusion_k in new_dict:
#             new_dict[fusion_k] = ori_dict[k]
#     elif fusion_k in new_dict:
#         new_dict[fusion_k] = ori_dict[k]
#         new_dict[fusion_k] = ori_dict[k]
# new_model.load_state_dict(new_dict)

In [ ]:
for k in ori_dict.keys():
    fusion_k = 'fusion.' + '.'.join(k.split('.')[1:])
    if fusion_k in new_dict:
        print(fusion_k)
        # new_dict[fusion_k] = ori_dict[k]

In [ ]:
new_model.save_pretrained('pruned_model_layer12', safe_serialization=False)
total_params = sum(p.numel() for p in new_model.parameters())
print(f"total parameters: {total_params}")